In [4]:
q1 = "How can i create my account to place order?"
q2 = "Do i need to sign up to place order?"

In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
v1 = model.encode(q1)
v2 = model.encode(q2)

In [6]:
v1.shape
v2.shape

(384,)

In [7]:
d  = "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process."
dv = model.encode(d)

In [8]:
v1.dot(dv)

np.float32(0.5912846)

In [9]:
v2.dot(dv)

np.float32(0.34497267)

In [10]:
import sys
import os

# 1. Direct Python to the root project folder
sys.path.append(os.path.abspath(os.path.join('..')))

In [11]:
from scripts.ingest import load_faq_data

documents = load_faq_data()

In [12]:
documents[10]

{'question': 'Do you offer gift wrapping services?',
 'answer': 'Yes, we offer gift wrapping services for an additional fee. During the checkout process, you can select the option to add gift wrapping to your order.',
 'id': 'N156mmfA'}

In [13]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

len(texts)

79

In [14]:
from tqdm.auto import tqdm

batch_size = 10
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)


  0%|          | 0/8 [00:00<?, ?it/s]

79

In [15]:
vectors[10].shape

(384,)

In [16]:
v1.dot(vectors[10])

np.float32(0.1843652)

In [17]:
scores=[]

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)


scores[:10]

[np.float32(0.59276354),
 np.float32(0.33685762),
 np.float32(0.42799753),
 np.float32(0.033179),
 np.float32(0.34425467),
 np.float32(0.10434541),
 np.float32(0.2555369),
 np.float32(0.15157577),
 np.float32(0.39817107),
 np.float32(0.27977812)]

In [18]:
import numpy as np

X = np.array(vectors)

In [19]:
scores1 = X.dot(v1)

scores1[:10]

array([0.59276354, 0.33685762, 0.4279976 , 0.03317901, 0.34425464,
       0.1043454 , 0.2555369 , 0.15157579, 0.39817107, 0.27977812],
      dtype=float32)

In [20]:
idx = np.argmax(scores1)

In [21]:
idx, scores1[idx]

(np.int64(16), np.float32(0.66570365))

In [22]:
documents[16]

{'question': 'Can I order without creating an account?',
 'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.',
 'id': 'REf0zOJF'}

In [23]:
top5 = np.argsort(-scores1)[:5]

In [24]:
scores1[top5]

array([0.66570365, 0.59276354, 0.46931157, 0.4279976 , 0.42172813],
      dtype=float32)

In [25]:
for idx in top5:
    print(scores1[idx])
    print(documents[idx])
    print()

0.66570365
{'question': 'Can I order without creating an account?', 'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.', 'id': 'REf0zOJF'}

0.59276354
{'question': 'How can I create an account?', 'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.", 'id': 'DLoeCO6R'}

0.46931157
{'question': 'Can I order by phone?', 'answer': 'Unfortunately, we do not accept orders over the phone. Please place your order through our website for a smooth and secure transaction.', 'id': 'SGvE1cKB'}

0.4279976
{'question': 'How can I track my order?', 'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.", 'id': 'aXRaBDBP'}

0.42172813
{'question':

In [26]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, documents)

In [27]:
vindex.search(v1, num_results=5)

[{'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.',
  'id': 'REf0zOJF'},
 {'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
  'id': 'DLoeCO6R'},
 {'question': 'Can I order by phone?',
  'answer': 'Unfortunately, we do not accept orders over the phone. Please place your order through our website for a smooth and secure transaction.',
  'id': 'SGvE1cKB'},
 {'question': 'How can I track my order?',
  'answer': "You can track your order by logging into your account and navigating to the 'Order History' section. There, you will find the tracking information for your shipment.",
  'id': 'aXRaBDBP'},
 {'question': 'Can I request a custom order or

In [28]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [29]:
from scripts.ingest import build_index

index = build_index(documents)

In [30]:
from scripts.rag_helper import RAGBase

assistant = RAGBase(
    index= index,
    llm_client=openai_client
)

In [31]:
query = 'How can i create my account?'
assistant.rag(query)

'To create an account, click on the **“Sign Up”** button in the **top right corner** of the website and follow the instructions to complete the registration process.'

In [32]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)

        return self.index.search(
            query_vector,
            num_results=num_results
        )

In [33]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client
)

In [34]:
query = 'How to register for an account?'
vector_assistant.rag(query)

'To register for an account, click the **“Sign Up”** button in the top right corner of the website and follow the instructions to complete the registration process.'

In [35]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    mode='ivf',
    db_path='../data/faq_vectors2.db'
)

In [36]:
vs_index.fit(vectors,documents)

In [37]:
query = "How to signup for an account?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [39]:
results

[{'question': 'How can I create an account?',
  'answer': "To create an account, click on the 'Sign Up' button on the top right corner of our website and follow the instructions to complete the registration process.",
  'id': 'DLoeCO6R'},
 {'question': 'Can I order without creating an account?',
  'answer': 'Yes, you can place an order as a guest without creating an account. However, creating an account offers benefits such as order tracking and easier future purchases.',
  'id': 'REf0zOJF'},
 {'question': 'Do you have a loyalty program?',
  'answer': 'Yes, we have a loyalty program where you can earn points for every purchase. These points can be redeemed for discounts on future orders. Please visit our website to learn more and join the program.',
  'id': 'fVQLiD79'},
 {'question': 'What should I do if my discount code is not working?',
  'answer': 'If your discount code is not working, please double-check the terms and conditions associated with the code. If the issue persists, cont

In [40]:
vs_index.close()